# 01 · Short-Term Memory: remembering the conversation

**Module 03 · Agent Memory & Retrieval**

In Module 01 you saw that the API has **no memory**. Each call stands alone, so *you* must resend the conversation. That list of messages is the agent's **short-term memory**.

In this notebook you will:
1. use the message list as memory,
2. see why it becomes a problem as it grows,
3. fix it by **trimming** and by **summarising**.

We continue with the same online shop from Module 02, and the assistant answers customers.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("Ready")

Ready


## A chat function that remembers
`history` is the memory. Every message, from the customer and from the assistant, is added to it, and the **whole list** is sent on every call.
The function also prints how many **input tokens** each call used, so you can watch the cost grow.

In [2]:
SYSTEM = {"role": "system", "content": "You are a friendly support assistant for an online shop. Keep answers to 2 sentences."}

def chat(history, user_text, keep=None):
    """Send one customer message. Both sides are saved in history."""
    history.append({"role": "user", "content": user_text})

    # Decide what to send to the model
    if keep is None:
        to_send = history                      # send the whole conversation
    else:
        system_message = history[0]            # always keep the system message
        last_messages = history[-keep:]        # plus only the most recent `keep` messages
        to_send = [system_message] + last_messages

    response = client.chat.completions.create(model=MODEL, messages=to_send)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    print(f"Customer : {user_text}")
    print(f"Assistant: {reply}")
    print(f"           (input tokens sent: {response.usage.prompt_tokens})\n")

In [3]:
history = [SYSTEM]

chat(history, "Hi, I'm Priya. I bought headphones, order A1001, and one earbud stopped working.")
chat(history, "I'd like a replacement if possible.")
chat(history, "What is my name, and which order are we talking about?")

Customer : Hi, I'm Priya. I bought headphones, order A1001, and one earbud stopped working.
Assistant: Hi Priya! I'm sorry to hear about the issue with your headphones. Please check our return policy, and if you're eligible, we can assist you with a replacement or refund.
           (input tokens sent: 51)



Customer : I'd like a replacement if possible.
Assistant: Of course! Please provide your order number and any details about the issue, and we'll initiate the replacement process for you.
           (input tokens sent: 102)



Customer : What is my name, and which order are we talking about?
Assistant: Your name is Priya, and we are discussing your order A1001 for the headphones.
           (input tokens sent: 147)



The assistant remembers, because the earlier messages were sent again. Look at the token counts: they **keep growing**, because each call carries the whole conversation.

## The problem: memory is not free
A long conversation means:
* **more cost**, since every old message is billed again on every call,
* **slower answers**, and
* eventually it **does not fit** in the model's context window (its maximum input size).

## Fix 1: trimming
Keep only the system message and the **last few messages**. Old messages are simply dropped. Here `keep=4` means the last 4 messages (2 turns).

In [4]:
history2 = [SYSTEM]

chat(history2, "Hi, I'm Priya and my order is A1001.", keep=4)
chat(history2, "How long does a replacement usually take?", keep=4)
chat(history2, "Can I pay for express shipping?", keep=4)
chat(history2, "What is my name?", keep=4)

Customer : Hi, I'm Priya and my order is A1001.
Assistant: Hi Priya! How can I assist you with your order A1001 today?
           (input tokens sent: 64)



Customer : How long does a replacement usually take?
Assistant: A replacement typically takes 5 to 7 business days to process and ship. You'll receive a notification once your replacement is on its way!
           (input tokens sent: 97)



Customer : Can I pay for express shipping?
Assistant: Yes, we offer express shipping options at checkout for faster delivery. Just select the express shipping option when you're completing your purchase!
           (input tokens sent: 101)



Customer : What is my name?
Assistant: I'm sorry, but I don't have access to your personal information, including your name. If you need assistance with something specific, feel free to ask!
           (input tokens sent: 106)



Trimming keeps the cost steady, but look at the last answer: **the assistant forgot Priya's name**, because that message was dropped. Trimming is cheap but forgetful.

## Fix 2: summarising
Instead of dropping old messages, we ask the model to **summarise** them in a couple of sentences, then keep the summary plus the last few messages. We lose detail but keep the important facts.

In [5]:
def summarize_older(history, keep_last=2):
    """Replace the older messages with a short summary. Keep the most recent ones as they are."""
    old = history[1:-keep_last]       # everything except the system message and the latest messages
    recent = history[-keep_last:]

    text = "\n".join(f"{m['role']}: {m['content']}" for m in old)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Summarise this conversation in 2 sentences. Keep names, order ids and requests:\n" + text}],
    )
    summary = response.choices[0].message.content

    return [history[0], {"role": "system", "content": "Summary of the conversation so far: " + summary}] + recent

history3 = [SYSTEM]

chat(history3, "Hi, I'm Priya and my order is A1001.")
chat(history3, "How long does a replacement usually take?")
chat(history3, "Can I pay for express shipping?")

print("Messages before:", len(history3))
history3 = summarize_older(history3)
print("Messages after :", len(history3))
print("\nThe summary:", history3[1]["content"], "\n")

chat(history3, "What is my name and order number?")

Customer : Hi, I'm Priya and my order is A1001.
Assistant: Hi Priya! How can I assist you with your order A1001 today?
           (input tokens sent: 42)



Customer : How long does a replacement usually take?
Assistant: A replacement typically takes 5 to 7 business days to process and ship. You'll receive a tracking number once your replacement is on its way!
           (input tokens sent: 75)



Customer : Can I pay for express shipping?
Assistant: Yes, express shipping options are available at checkout. You can select your desired shipping method for faster delivery during the ordering process.
           (input tokens sent: 119)

Messages before: 7


Messages after : 4

The summary: Summary of the conversation so far: Priya inquired about her order A1001, specifically asking how long a replacement would take. The response indicated that replacements usually take 5 to 7 business days to process and ship, after which she would receive a tracking number. 



Customer : What is my name and order number?
Assistant: Your name is Priya, and your order number is A1001. If you have any more questions about your order, feel free to ask!
           (input tokens sent: 136)



Now the assistant knows the name and order, from a **summary** instead of the full history.

## Recap
| Approach | Cost | What is lost |
|---|---|---|
| Send everything | Grows every turn | Nothing, until it no longer fits |
| **Trim** (last N messages) | Steady | Anything older than N |
| **Summarise** | Steady | Fine detail, but key facts stay |

All of these are **short-term memory**: they live only while the conversation lasts. When the program stops, everything is gone.

Next: how to find the *right* information from a large pile of text, using **embeddings**.